In [2]:
import sys
sys.path.append("/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/")
import pandas as pd
import numpy as np
or_data = pd.read_csv("../../Data/data_processed/Original_data/integrated_data_jobim.csv", index_col = 0)
metadata_path = "../../Data/data_processed/Original_data/feature_metadata.json"
or_data.head()

,Number_of_Prior_Therapies,ORR,Benefit,Cohort,Sex,MSKCC,Sarc,Arm,OS_CNSR,Rhab,...,ENSG00000198899.2,ENSG00000198938.2,ENSG00000198840.2,ENSG00000212907.2,ENSG00000198886.2,ENSG00000210176.1,ENSG00000198786.2,ENSG00000198695.2,ENSG00000210194.1,ENSG00000198727.2
Patient_ID,,,,,,,,,,,,,,,,,,,,,
G138701_RCCBMS-00020-T_v1_RNA_OnPrem,2.0,SD,CB,CM-010,M,FAVORABLE,0.0,NIVOLUMAB,1.0,0.0,...,32.02459,32.43928,28.87370,29.40491,34.00210,21.13002,31.87140,31.90028,21.26386,28.67162
G138701_RCCBMS-00097-T_v1_RNA_OnPrem,2.0,PR,CB,CM-010,F,FAVORABLE,0.0,NIVOLUMAB,1.0,0.0,...,34.67436,34.35974,33.35211,34.23558,34.75711,21.13002,33.69635,33.57917,21.26386,33.96374
G138701_RCCBMS-00141-T_v1_RNA_OnPrem,1.0,PR,CB,CM-010,F,POOR,0.0,NIVOLUMAB,0.0,0.0,...,33.92360,33.17470,32.86634,32.95676,34.37301,21.13002,33.46758,32.99777,21.26386,33.29667
G138701_RCCBMS-00099-T_v1_RNA_OnPrem,3.0,PD,NCB,CM-010,M,FAVORABLE,0.0,NIVOLUMAB,1.0,0.0,...,33.25229,32.88098,33.23512,34.18982,34.09013,21.13002,33.52007,33.08947,21.26386,32.90812
G138701_RCCBMS-00163-T_v1_RNA_OnPrem,2.0,SD,ICB,CM-010,M,INTERMEDIATE,0.0,NIVOLUMAB,0.0,0.0,...,27.94141,29.16648,28.87370,29.40491,32.73904,21.13002,30.58496,28.49402,21.26386,28.67162


In [5]:
meta_data = pd.read_csv("../../Data/Data_Braun/metadata_Braun.csv")

In [16]:
a = or_data.index.tolist()

In [23]:
rna_seq =meta_data[meta_data["RNA_ID"].isin(a)]

In [31]:
rna_seq.columns[11:]

Index(['SUBJID', 'Cohort', 'Arm', 'MAF_Tumor_ID', 'MAF_Normal_ID', 'CNV_ID',
       'RNA_ID', 'CD8_IF_ID', 'Sex', 'Age',
       ...
       'TSC1', 'USP32', 'VHL', 'WNT8A', 'ZNF800', 'Angio', 'Teff', 'Myeloid',
       'Javelin', 'Merck18'],
      dtype='object', length=121)

In [2]:
from SynOmics.metrics.fidelity.utils import extract_metadata

metadata = extract_metadata(or_data, metadata_path)

In [3]:
from utils import check_duplicates
num_cols = metadata.get("numerical")
num_df = or_data[num_cols]
duplicated_dict = check_duplicates(num_df)

subset = or_data[num_cols]
dup_mask_numeric = subset.T.duplicated(keep='first')
keep_mask_all = pd.Series(True, index=or_data.columns)
keep_mask_all.loc[dup_mask_numeric[dup_mask_numeric].index] = False
# DataFrame sau khi loại duplicate chỉ trong num_cols
df_cleaned_identical = or_data.loc[:, keep_mask_all]

print(df_cleaned_identical.shape)

# Get removed features (all columns)
removed_features = keep_mask_all[~keep_mask_all].index.tolist()
identical_df = or_data[removed_features]

Grouping genes with identical expression values...


Processing duplicates: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2085/2085 [00:00<00:00, 17513.96it/s]


(310, 39294)


In [4]:
from SynOmics.metrics.fidelity.utils import PreprocessingForValidation
cleaned_metadata = extract_metadata(df_cleaned_identical, metadata_path)
missing_indicators_df = df_cleaned_identical[cleaned_metadata.get("missing_categorical")]
pre_pro_ = PreprocessingForValidation(
        data = df_cleaned_identical,
        target_col = "Benefit",
        output_dir = ".",
        is_synthetic = False,
        ordinal_cat_columns = cleaned_metadata.get("ordinal_categorical"),
        dummy_cat_columns = cleaned_metadata.get("dummy_categorical"),
        numerical_columns = cleaned_metadata.get("numerical"),
        scaler  = "minmax",
        n_neighbors = 5
)
preprocessed_data = pre_pro_.fit()
preprocessed_data = pd.concat([preprocessed_data, missing_indicators_df], axis = 1)

2025-09-15 10:53:49 - INFO - Ordinal encoding 2 features
2025-09-15 10:53:50 - INFO - Dummy encoding 10 features
2025-09-15 10:53:50 - INFO - Using scaler: minmax
2025-09-15 10:53:50 - INFO - Normalizing 39275 numerical features


In [5]:
preprocessed_data.to_csv("proprocessed_or_data.csv", index = True)